# ============================================================
# STEP 2C-EXT — EXTENDED TURKISH LINGUISTIC FEATURES (TUMC)
# ============================================================
# ADDS 5 new score features to the existing 20 Turkish features
# (does NOT rebuild them). These are the genuinely novel scores
# not captured by the original Step 2C:
#
#   1. tr_phishing_vocab_score   — weighted phishing-lexicon match
#   2. tr_transliteration_score  — ASCII-folded Turkish brand/word
#        detection (the key evasion: garantibankasi for garantibankası)
#   3. tr_brand_impersonation_score — min edit-distance to a TR brand
#        among non-exact matches (lookalike detection)
#   4. tr_semantic_urgency_score — weighted urgency-lexicon match
#   5. tr_scam_vocab_score       — weighted scam-lexicon match
#
# Reads features_full_final.csv (or v11) and writes a +5 version.
# Includes MI + correlation-with-existing check so redundant
# features can be pruned honestly.
#
# Output: features_full_final.csv, turkish_extended_feature_names_final.pkl
# ============================================================

In [ ]:
!pip install tldextract
import os, re, pickle, warnings
from urllib.parse import urlparse, unquote
import numpy as np
import pandas as pd
import tldextract
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings("ignore")
SEED = 42
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
for cand in ["features_full_final.csv","features_full_final.csv"]:
    INPUT_FILE = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT_FILE): break
OUTPUT_CSV = os.path.join(DATA_DIR, "features_full_final.csv")
OUTPUT_PKL = os.path.join(DATA_DIR, "turkish_extended_feature_names_final.pkl")

# Import the curated lexicons (place turkish_lexicons.py alongside)
import sys; sys.path.insert(0, DATA_DIR); sys.path.insert(0, ".")
from turkish_lexicons import (TR_PHISHING, TR_SCAM, TR_URGENCY,
                              TR_BRANDS, TR_TRANSLIT, TR_BRANDS_WITH_TRCHARS)
# Sanitise: dedup brands, drop generic/too-short match-prone tokens
from lexicon_cleaner import clean_lexicon, clean_brands
TR_PHISHING = clean_lexicon(TR_PHISHING)
TR_SCAM     = clean_lexicon(TR_SCAM)
TR_URGENCY  = clean_lexicon(TR_URGENCY)
TR_BRANDS   = clean_brands(TR_BRANDS)
print(f"  cleaned lexicons → phishing {len(TR_PHISHING)}, scam {len(TR_SCAM)}, "
      f"urgency {len(TR_URGENCY)}, brands {len(TR_BRANDS)}")

NEW_FEATURES = [
    "tr_phishing_vocab_score",
    "tr_transliteration_score",
    "tr_brand_impersonation_score",
    "tr_semantic_urgency_score",
    "tr_scam_vocab_score",
]

_extract = tldextract.TLDExtract()
try: _extract("test.com.tr")
except Exception: _extract = tldextract.TLDExtract(suffix_list_urls=())

def tokenize(u):
    return [t for t in re.split(r"[^a-zçğıöşü0-9]+", unquote(str(u).lower())) if len(t) >= 2]

def translit_fold(s):
    return "".join(TR_TRANSLIT.get(c, c) for c in s)

def lex_score(tokens, joined, lexicon):
    """Weighted, length-normalised lexicon match score in [0,1].
    Token-aware: a single-word term must match a WHOLE token (so 'son'
    does not fire inside 'reason'); multiword/hyphenated terms match as
    substrings (they are specific enough). Avoids substring-everywhere."""
    if not tokens: return 0.0
    token_set = set(tokens)
    hit = 0
    for term, w in lexicon.items():
        if " " in term or "-" in term:
            if term.replace("-", "") in joined or term in joined:
                hit += w
        else:
            # exact token, or token startswith term (Turkish agglutination:
            # 'hesap' matches 'hesabiniz'); require term length>=5 for prefix
            if term in token_set or (len(term) >= 5 and
                                     any(tok.startswith(term) for tok in tokens)):
                hit += w
    return round(min(hit / 10.0, 1.0), 4)

def levenshtein(a, b, cap=4):
    """Bounded edit distance; returns cap+1 if clearly beyond cap (fast)."""
    if abs(len(a)-len(b)) > cap: return cap+1
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[-1]+1, prev[j-1]+(ca!=cb)))
        prev = cur
        if min(prev) > cap: return cap+1
    return prev[-1]

# Pre-fold brand list once
BRANDS_FOLDED = [(b, translit_fold(b)) for b in TR_BRANDS]

def extract_extended(url):
    u = unquote(str(url).lower())
    ext = _extract(url if str(url).startswith("http") else "http://"+str(url))
    domain = (ext.domain or "").lower()
    tokens = tokenize(u)
    joined = u.replace("/", "").replace(".", "")
    f = {}

    # 1 & 4 & 5: weighted lexicon scores
    f["tr_phishing_vocab_score"]  = lex_score(tokens, u, TR_PHISHING)
    f["tr_semantic_urgency_score"] = lex_score(tokens, u, TR_URGENCY)
    f["tr_scam_vocab_score"]      = lex_score(tokens, u, TR_SCAM)

    # 2: transliteration score — does the (ASCII) URL contain an
    # ASCII-folded form of a brand whose canonical name has Turkish
    # chars, written WITHOUT those chars? (e.g. "yapikredi" for
    # "yapıkredi") — the evasion that dodges Turkish-char detection.
    folded_url = translit_fold(u)
    translit_hits = 0
    for brand_tr in TR_BRANDS_WITH_TRCHARS:
        folded_brand = translit_fold(brand_tr)
        # URL contains the ASCII-folded brand but NOT the true TR form
        if folded_brand in folded_url and brand_tr not in u:
            translit_hits += 1
    f["tr_transliteration_score"] = round(min(translit_hits / 2.0, 1.0), 4)

    # 3: brand impersonation — domain is a near-miss of a TR brand
    # (close edit distance but NOT an exact match → lookalike)
    best = 99
    dom_folded = translit_fold(domain)
    for orig, folded in BRANDS_FOLDED:
        if domain == orig or folded == dom_folded and orig == domain:
            best = 0; break                      # exact/legit — not impersonation
        d = levenshtein(dom_folded, folded, cap=3)
        if 0 < d < best:
            best = d
    # score: 1.0 if edit distance 1, decaying; 0 if exact or far
    if best == 0 or best > 3:
        f["tr_brand_impersonation_score"] = 0.0
    else:
        f["tr_brand_impersonation_score"] = round(1.0 - (best-1)/3.0, 4)

    return {k: f[k] for k in NEW_FEATURES}

print("="*70)
print("STEP 2C-EXT — EXTENDED TURKISH FEATURES (+5)")
print("="*70)

df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"\n[1] Loaded {len(df):,} rows, {len(df.columns)} cols from {os.path.basename(INPUT_FILE)}")

print("\n[2] Validation:")
tests = [
    "https://garanti.com.tr/bireysel/giris",           # legit bank
    "http://garantibankasi-giris.xyz/dogrulama",        # translit + phishing
    "http://garant1-online.tk/hesap-dogrula-acil",      # impersonation + urgency
    "https://trendyol.com/kampanya/indirim",            # legit ecom
    "http://kazandiniz-cekilis.info/odul-bedava",       # scam
]
for t in tests:
    ft = extract_extended(t)
    print(f"  {t[:50]}")
    print(f"    phish={ft['tr_phishing_vocab_score']:.2f} "
          f"translit={ft['tr_transliteration_score']:.2f} "
          f"brandimp={ft['tr_brand_impersonation_score']:.2f} "
          f"urgency={ft['tr_semantic_urgency_score']:.2f} "
          f"scam={ft['tr_scam_vocab_score']:.2f}")

print(f"\n[3] Extracting for {len(df):,} URLs (offline) ...")
records = []
for i, url in enumerate(df["url"]):
    records.append(extract_extended(url))
    if (i+1) % 100_000 == 0: print(f"    {i+1:>9,}/{len(df):,}")
ext_df = pd.DataFrame(records)
print(f"  matrix {ext_df.shape}, nulls {ext_df.isnull().sum().sum()}")

# Merge
full = pd.concat([df.reset_index(drop=True), ext_df.reset_index(drop=True)], axis=1)
full.to_csv(OUTPUT_CSV, index=False)
pickle.dump(NEW_FEATURES, open(OUTPUT_PKL,"wb"))
print(f"\n[4] Saved {len(df.columns)+5} features → {OUTPUT_CSV}")

# Class rates
ext_df["class_final"] = df["class_final"].values
print(f"\n[5] New-feature means by class:")
print(ext_df.groupby("class_final")[NEW_FEATURES].mean().round(3).to_string())

# MI (5-class) + redundancy check vs existing Turkish features
print(f"\n[6] MI (5-class) of new features:")
mi = mutual_info_classif(ext_df[NEW_FEATURES].values, df["class_enc"].values, random_state=SEED)
for f_, m_ in sorted(zip(NEW_FEATURES, mi), key=lambda t:-t[1]):
    print(f"    {f_:<32s}: {m_:.4f}")

print(f"\n[7] Max |correlation| of each new feature with EXISTING features")
print("    (high corr = redundant, candidate to drop):")
existing_tr = [c for c in df.columns if c.startswith("tr_") or c in
               ["vowel_harmony_score","langid_tr_confidence","is_turkish_dominant",
                "has_tr_bank_term","has_tr_gov_term","has_tr_ecommerce_term"]]
for nf in NEW_FEATURES:
    if existing_tr:
        cors = df[existing_tr].apply(lambda col: np.corrcoef(col.fillna(0), ext_df[nf])[0,1]
                                      if col.std()>0 else 0)
        mx = cors.abs().max(); who = cors.abs().idxmax()
        flag = " ⚠ REDUNDANT" if mx>0.85 else ""
        print(f"    {nf:<32s}: max|r|={mx:.3f} with {who}{flag}")

print(f"""
{'='*70}
STEP 2C-EXT COMPLETE
{'='*70}
  Review [6] and [7]: keep features with non-trivial MI AND
  max|r| < 0.85 vs existing. Drop any flagged redundant.
  Honest reporting: state which of the 5 survived screening.
  Next → re-run Step 2E selection including survivors.
{'='*70}
""")

  cleaned lexicons → phishing 154, scam 105, urgency 47, brands 193
STEP 2C-EXT — EXTENDED TURKISH FEATURES (+5)

[1] Loaded 1,239,308 rows, 125 cols from features_full_final.csv

[2] Validation:
  https://garanti.com.tr/bireysel/giris
    phish=0.30 translit=0.00 brandimp=0.00 urgency=0.00 scam=0.00
  http://garantibankasi-giris.xyz/dogrulama
    phish=0.90 translit=0.00 brandimp=0.00 urgency=0.00 scam=0.00
  http://garant1-online.tk/hesap-dogrula-acil
    phish=0.50 translit=0.00 brandimp=0.00 urgency=0.00 scam=0.00
  https://trendyol.com/kampanya/indirim
    phish=0.00 translit=0.00 brandimp=0.00 urgency=0.00 scam=0.20
  http://kazandiniz-cekilis.info/odul-bedava
    phish=0.00 translit=0.00 brandimp=0.00 urgency=0.00 scam=0.90

[3] Extracting for 1,239,308 URLs (offline) ...
      100,000/1,239,308
      200,000/1,239,308
      300,000/1,239,308
      400,000/1,239,308
      500,000/1,239,308
      600,000/1,239,308
      700,000/1,239,308
      800,000/1,239,308
      900,000/1,23